![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/orders.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
(spark.read.table("bronze").filter("topic = 'orders'")).count()

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

batch_total = (spark.read
                        .table("bronze")
                        .filter("topic = 'orders'")
                        .select(F.from_json(F.col("value").cast("string"), json_schema).alias("data"))
                        .select("data.*")
                        .drop_duplicates(['order_id', 'order_timestamp'])
                        .count()
               )
print(batch_total)

In [0]:
df_deduped = (
    spark.readStream
            .table("bronze")
            .filter("topic = 'orders'")
            .select(F.from_json(F.col("value").cast("string"), json_schema).alias("data"))
            .select("data.*")
            .withWatermark("order_timestamp", "30 seconds")
            .drop_duplicates(['order_id', 'order_timestamp'])

)

In [0]:
def upsert_data(microBatchDF, batchId):
    microBatchDF.createOrReplaceTempView("orders_microbatch")

    sql_query = """
        merge into orders_silver a
        using orders_microbatch b
        on a.order_id = b.order_id and a.order_timestamp = b.order_timestamp
        when not matched then insert *
    """
    microBatchDF.sparkSession.sql(sql_query)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS orders_silver
(order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>)

In [0]:
query = (df_deduped.writeStream
                    .foreachBatch(upsert_data)
                    .option("checkpointLocation", f"{bookstore.checkpoint_path}/orders_silver")
                    .trigger(availableNow=True)
                    .start()
         )
query.awaitTermination()

In [0]:
streaming_total = (spark.read.table("orders_silver").count())

In [0]:
assert streaming_total == batch_total , "counts are not matching"
print("test passed, counts are matching")